In [1]:
import pandas as pd
import re
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
from dash.dependencies import Input, Output, State

load_dotenv()

# --- Database Setup ---
engine = create_engine(f"mysql+pymysql://root:@localhost:3306/rxnorm?charset=utf8mb4")

# --- Helper Functions ---
def extract_name(df):
    """Extracts bracketed text, titles it, and removes duplicates."""
    if df.empty:
        return df
    df["Product_Name"] = df["STR"].str.extract(r'\[(.*?)\]')
    df["Product_Name"] = df["Product_Name"].fillna("Generic")
    df["Product_Name"] = df["Product_Name"].str.title()
    df = df.drop_duplicates(subset=["Product_Name"])
    return df

def Searchbar(term):
    sql = text("SELECT RXCUI, STR FROM RXNCONSO WHERE STR LIKE :term AND TTY IN ('DP')")
    with engine.connect() as conn:
        df = pd.read_sql(sql, conn, params={'term': f'%{term}%'})
    return extract_name(df)

def Fetch_Ingredients(ID):
    """
    Fetches ingredients and parses the strength (MG) from the name string.
    """
    query = text("""
        SELECT r.RXCUI2 as Ingredient_ID, c.STR as Full_Ingredient
        FROM RXNCONSO c
        JOIN RXNREL r ON c.RXCUI = r.RXCUI2
        WHERE r.RXCUI1 = :id AND c.TTY = "SCDC"
        GROUP BY Ingredient_ID, Full_Ingredient;
    """)
    
    with engine.connect() as conn:
        df = pd.read_sql(query, conn, params={"id": ID})
    
    # Parse the 'Strength' out of the RxNorm SCDC string (e.g., "Amoxicillin 500 MG")
    parsed_data = []
    for _, row in df.iterrows():
        match = re.search(r"(.+?)\s+(\d+(?:\.\d+)?)\s+MG", row["Full_Ingredient"], re.IGNORECASE)
        if match:
            parsed_data.append({
                "Ingredient_ID": row["Ingredient_ID"],
                "Ingredient": match.group(1).strip(),
                "Concentration": match.group(2)
            })
        else:
            parsed_data.append({
                "Ingredient_ID": row["Ingredient_ID"],
                "Ingredient": row["Full_Ingredient"],
                "Concentration": "N/A"
            })
    
    return pd.DataFrame(parsed_data)

def Fetch_Dose_Form(ID):
    query = text("SELECT c.STR FROM RXNCONSO c JOIN RXNREL r ON c.RXCUI = r.RXCUI2 WHERE r.RXCUI1 = :id AND c.TTY = 'DF'")
    with engine.connect() as conn:
        res = pd.read_sql(query, conn, params={'id': ID})
    return res["STR"].iloc[0] if not res.empty else "Not specified"

def fetch_generic_name(ID):
    query = text("SELECT c.STR FROM RXNCONSO c JOIN RXNREL r ON c.RXCUI = r.RXCUI2 WHERE r.RXCUI1 = :id AND c.TTY = 'SCD'")
    with engine.connect() as conn:
        res = pd.read_sql(query, conn, params={'id': ID})
    return res["STR"].iloc[0] if not res.empty else "N/A"

def Fetch_Exact_Drugs(Ing_lst,ID):
    s = ""
    for i,j in enumerate(Ing_lst):
        if i == (len(Ing_lst) - 1):
            s+="r1.RXCUI1 = "+j
        else:
            s+="r1.RXCUI1 = "+j+" or "
            
    query = f"""
    WITH base AS (
        SELECT r2.RXCUI as ID, r2.STR as DP, r1.RXCUI1 as Ingredient_ID
        FROM RXNREL r1
        JOIN RXNCONSO r2
        ON r1.RXCUI2 = r2.RXCUI
        WHERE ({s}) and r2.TTY = "DP"
    ),
    keys_all AS (
        SELECT ID
        FROM base
        GROUP by ID
        HAVING COUNT(DISTINCT Ingredient_ID) = {len(Ing_lst)}
    )
    SELECT b.ID,b.DP
    FROM base b
    JOIN keys_all k
    ON b.ID = k.ID
    WHERE b.Id != {ID}
    GROUP BY b.ID, b.DP
    """
    
    res = pd.read_sql(query, engine)
    
    lst = []
    drp = []
    for j,i in enumerate(res["DP"]): 
        if "[" in i:
            d = i.split("[")
            lst.append(d[-1][:-1])
        else:
            lst.append("Generic")
            drp.append(j)

    res["Product_Name"] = lst
    
    Product = []
    for j,i in enumerate(res["Product_Name"]):
        if i.lower() in Product:
            drp.append(j)
        else:
            Product.append(i.lower())
    res = res.drop(drp)
    res = res.reset_index(drop=True)
    return res
    

In [2]:
Searchbar("Tylenol")

,RXCUI,STR,Product_Name
0,209387,ACETAMINOPHEN 325 mg ORAL TABLET [TYLENOL Regu...,Tylenol Regular Strength
4,209459,"ACETAMINOPHEN 500 mg ORAL TABLET, FILM COATED ...",Tylenol Extra Strength
18,209459,"ACETAMINOPHEN 500 mg ORAL TABLET, COATED [TYLE...","Tylenol Extra Strength, Cvp Health"
19,209459,"ACETAMINOPHEN 500 mg ORAL TABLET, COATED [TYLE...","Tylenol Extra Strength, Travel Basix"
26,209459,ACETAMINOPHEN 500 mg ORAL TABLET [Tylenol Extr...,Tylenol Extra Strength Caplet
27,247324,ACETAMINOPHEN 500 mg / DEXTROMETHORPHAN HYDROB...,Tylenol
28,247324,DEXTROMETHORPHAN HYDROBROMIDE 15 mg / ACETAMIN...,Extra Strength Tylenol Severe Cough Plus Sore ...
29,250912,"MENTHOL, UNSPECIFIED FORM 10 mg in 1 g / LIDOC...",Tylenol Precise Cooling Pain Relieving
31,828555,ACETAMINOPHEN 160 mg in 5 mL ORAL SUSPENSION [...,Childrens Tylenol
34,828555,ACETAMINOPHEN 160 mg in 5 mL ORAL SUSPENSION [...,Infants Tylenol


In [7]:
Fetch_Ingredients(1092378)
ingredients = Fetch_Ingredients(1092378)
Fetch_Exact_Drugs(ingredients["Ingredient_ID"].tolist(), 1092378)

,ID,DP,Product_Name
0,1092189,ACETAMINOPHEN 500 mg / DIPHENHYDRAMINE HYDROCH...,Pain Relief PM Extra Strength
1,1092189,ACETAMINOPHEN 500 mg / DIPHENHYDRAMINE HYDROCH...,Pain Relief PM
2,1092189,ACETAMINOPHEN 500 mg / DIPHENHYDRAMINE HYDROCH...,CAREALL Non-Aspirin PM Extra Strength
3,1092189,DIPHENHYDRAMINE HYDROCHLORIDE 25 mg / ACETAMIN...,CounterAct PM
4,1092189,ACETAMINOPHEN 500 mg / DIPHENHYDRAMINE HYDROCH...,ACETAMINOPHEN PM
5,1092189,ACETAMINOPHEN 500 mg / DIPHENHYDRAMINE HYDROCH...,Pain Reliever PM
6,1092189,ACETAMINOPHEN 500 mg / DIPHENHYDRAMINE HYDROCH...,Pain Reliever PM Extra strength
7,1092189,DIPHENHYDRAMINE HYDROCHLORIDE 25 mg / ACETAMIN...,Vicks ZzzQuil Night Pain
8,1092189,ACETAMINOPHEN 500 mg / DIPHENHYDRAMINE HYDROCH...,EXTRA STRENGTH PAIN RELIEF PM
9,1092189,ACETAMINOPHEN 500 mg / DIPHENHYDRAMINE HYDROCH...,EXTRA STRENGTH PAIN RELIEVER PM


NameError: name 'ingredients' is not defined